Setup & libraries installation

In [2]:
import numpy as np

Defining variables

In [16]:
ha, hb, hc, hd, he, hf, hg = 1492, 1492, 1491, 1490, 1490, 1489, 1486 # (m)
h_ref = hg

loss_t_section, loss_inlet = 1, 0.5 # (/)
water_density, dyn_visc, kin_visc =  998, 1.307*10*-3, 1.307*10*-6
Qdesired = 1.67e-4 # m3 /s ------- =10 l / min
ab, bc, cd, de, ef, fg = 10,10,10,10,10,40 # (m)
l_hose = 15 #m
g = 9.81

#lists
heights = [hb, hc, hd, he, hf, hg] #ha hebben we hier niet nodig
distances = [ab, bc, cd, de, ef, fg]

#garden hose 3/4 inch
D_hose = 0.019 #(m)
e_hose = 0.0015 *10e-3 #(m)
A_hose = D_hose**2 * np.pi / 4
U_hose = Qdesired / A_hose

#hdpe mainline 3/4 inch
D_main = 0.019 #(m)
e_main = 0.000003 #(m)
A_main = D_main**2 * np.pi / 4
U_main = Qdesired / A_main

In [12]:
print('Speed hose: ' + str(round(U_hose, 2)) + ' m/s \nSpeed mainline: ' + str(round(U_main,2)) + ' m/s')

Speed hose: 0.59 m/s 
Speed mainline: 0.59 m/s


Often used functions

In [13]:
def Reynold(Q, D):
    Re = (4 * Q) / (np.pi * D * kin_visc)
    return(Re)


def WhiteColeBrook(Re, f, e, D):
    fnew = 1 / (-2 * np.log10 (2.51 / (Re * np.sqrt(f)) + e / (3.7 * D)))**2
    return(fnew)

def CalculateHead(n, h_end,  f_main, f_hose): #amount of segments, height of endpoint
    main_losses = 0
    
    for i in range(n):
        main_losses += (f_main * distances[i]  * U_main**2) / (D_main * 2 * g)
      #  print('Main losses segment ' + str(i) + ': ' + str(round(main_losses, 2)) + ' m')

    hose_losses = (f_hose * l_hose  * U_hose**2) / (D_hose * 2 * g)
    local_losses = (loss_inlet + loss_t_section * n) * (Qdesired**2) / (A_main**2 * 2 * g) # #n T-sections

    
  #  print('Hose losses ' + str(round(hose_losses, 2)) + ' m')
   # print('Local losses ' + str(round(local_losses, 2)) + ' m')


    total_losses = main_losses + hose_losses + local_losses
    total_head = h_end + Qdesired**2 / (A_hose**2 * 2 * g) + total_losses
    return(total_head)


Calculations

In [14]:
Re_hose = Reynold(Qdesired, D_hose)
Re_main = Reynold(Qdesired, D_main)

Re_hose, Re_main

(-0.0001427072873265226, -0.0001427072873265226)

In [15]:


f_hose_guess_1 = 0.020
f_hose_guess_2 = WhiteColeBrook(Re_hose, f_hose_guess_1, e_hose, D_hose)
f_hose_guess_3 = WhiteColeBrook(Re_hose, f_hose_guess_2, e_hose, D_hose)
f_hose_guess_4 = WhiteColeBrook(Re_hose, f_hose_guess_3, e_hose, D_hose)
f_hose = WhiteColeBrook(Re_hose, f_hose_guess_4, e_hose, D_hose)
print(f_hose)

f_main_guess_1 = 0.020
f_main_guess_2 = WhiteColeBrook(Re_main, f_main_guess_1, e_main, D_main)
f_main_guess_3 = WhiteColeBrook(Re_main, f_main_guess_2, e_main, D_main)
f_main_guess_4 = WhiteColeBrook(Re_main, f_main_guess_3, e_main, D_main)
f_main = WhiteColeBrook(Re_main, f_main_guess_4, e_main, D_main)
print(f_main)


nan
nan


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_17392\1164592614.py:7: RuntimeWarning: invalid value encountered in log10
  fnew = 1 / (-2 * np.log10 (2.51 / (Re * np.sqrt(f)) + e / (3.7 * D)))**2


Voorlopig uitgaande van een tuinslang zonder speciaal opzetstuk

In [9]:
i = 0
points = ['B', 'C', 'D', 'E', 'F', 'G']
for height in heights:
    z = CalculateHead(1, height, f_hose, f_main)
    print('Neccesary head at pointA for' + points[i] + '= ' + str(round(z - ha, 2)))
    i += 1



Neccesary head at pointA forB= 0.73
Neccesary head at pointA forC= -0.27
Neccesary head at pointA forD= -1.27
Neccesary head at pointA forE= -1.27
Neccesary head at pointA forF= -2.27
Neccesary head at pointA forG= -5.27


De tank moet dus niet op een verhoog staan om te werken

Volgende stappen:
1) e_... vervangen door k



